In [3]:
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import piexif
import shutil

# ============================================================
# Roots (recursive search)
# ============================================================

roots = [
    Path('DATA/DATA/images')
]

# ============================================================
# Output directory
# ============================================================

out_dir = Path('DATA/DATA/images/IOEO_intervals')
out_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# IOEO date boundaries
# ============================================================

ioeo_dates = [
    datetime.fromisoformat("2025-01-23 13:00:00"),
    datetime.fromisoformat("2025-02-19 13:00:00"),
]

# ============================================================
# Loop through roots
# ============================================================

for root in roots:

    # Recursive TIFF search
    files = root.rglob("*.tiff")

    for filepath in files:

        filename = filepath.name
        folder_str = str(filepath.parent).lower()

        # ----------------------------------------------------
        # Skip unwanted folders
        # ----------------------------------------------------
        if "timex" in folder_str or "raw" in folder_str:
            continue

        # ----------------------------------------------------
        # Extract camera
        # ----------------------------------------------------
        if ".c1." in filename:
            cam = "c1"
        elif ".c2." in filename:
            cam = "c2"
        else:
            continue

        # ----------------------------------------------------
        # Extract datetime from filename
        # ----------------------------------------------------
        parts = filename.split(".")

        try:
            epoch_time = float(parts[0])
            file_time = datetime.fromtimestamp(epoch_time, tz=timezone.utc)
        except Exception:
            continue

        # ----------------------------------------------------
        # Find IOEO interval
        # ----------------------------------------------------
        for k in range(len(ioeo_dates) - 1):

            start_date = ioeo_dates[k]
            end_date = ioeo_dates[k + 1]

            # Make naive datetime comparison
            file_time_naive = file_time.replace(tzinfo=None)

            if start_date <= file_time_naive < end_date:

                # Interval folder name
                folder_name = (
                    f"{start_date.strftime('%Y_%m_%d')}"
                    f"_to_"
                    f"{end_date.strftime('%Y_%m_%d')}"
                )

                # Final destination
                dest_folder = out_dir / folder_name / cam
                dest_folder.mkdir(parents=True, exist_ok=True)

                # Output PNG filename
                png_name = filepath.stem + ".png"
                dest_file = dest_folder / png_name

                try:

                    # ----------------------------------------
                    # Open TIFF
                    # ----------------------------------------
                    img = Image.open(filepath)

                    # ----------------------------------------
                    # Preserve EXIF metadata if present
                    # ----------------------------------------
                    exif_bytes = None

                    try:
                        exif_data = img.info.get("exif")

                        if exif_data:
                            exif_dict = piexif.load(exif_data)
                            exif_bytes = piexif.dump(exif_dict)

                    except Exception as exif_error:
                        print(f"EXIF warning for {filename}: {exif_error}")

                    # ----------------------------------------
                    # Convert image mode if needed
                    # PNG does not support some TIFF modes
                    # ----------------------------------------
                    if img.mode not in ("RGB", "RGBA", "L"):
                        img = img.convert("RGB")

                    # ----------------------------------------
                    # Save PNG
                    # ----------------------------------------
                    if exif_bytes:
                        img.save(dest_file, format="PNG", exif=exif_bytes)
                    else:
                        img.save(dest_file, format="PNG")

                    print(f"Saved: {dest_file}")

                except Exception as e:
                    print(f"Failed processing {filepath}")
                    print(e)

                break

print("Done sorting by IOEO date intervals.")

Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c2/1738517400.Sun.Feb.02_17_30_00.GMT.2025.CACO04.c2.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c1/1738758600.Wed.Feb.05_12_30_00.GMT.2025.CACO04.c1.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c1/1738459800.Sun.Feb.02_01_30_00.GMT.2025.CACO04.c1.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c2/1739883600.Tue.Feb.18_13_00_00.GMT.2025.CACO04.c2.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c1/1737846000.Sat.Jan.25_23_00_00.GMT.2025.CACO04.c1.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c1/1739226600.Mon.Feb.10_22_30_00.GMT.2025.CACO04.c1.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c2/1737741600.Fri.Jan.24_18_00_00.GMT.2025.CACO03.c2.ras.png
Saved: DATA/DATA/images/IOEO_intervals/2025_01_23_to_2025_02_19/c2/1738690200.Tue.Feb.04_17_30_00.GMT.2025.CACO04.c2.ras.png
